# WikiTrend LightGBM Prediction Inspection

This notebook inspects the read-only Gold outputs created by `score_lightgbm.py`. It covers manifests, schemas, prediction rows, fallback behavior, research rankings, actual-vs-prediction metrics, and model metadata.

It does not retrain the model or modify any data.

## 1. Project Setup

Run this notebook from the repository root or the `notebooks` directory with the `wikitrend` kernel.

In [ ]:
from pathlib import Path
import json

import duckdb
import pandas as pd
import pyarrow.dataset as ds
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'data').exists():
    raise RuntimeError('Start Jupyter from the WikiTrend project or notebooks directory.')

PREDICTION_ROOT = PROJECT_ROOT / 'data' / 'gold' / 'lightgbm_predictions'
PREDICTIONS_DIR = PREDICTION_ROOT / 'predictions'
TOP_PAGES_DIR = PREDICTION_ROOT / 'research_top_pages'
METRICS_DIR = PREDICTION_ROOT / 'metrics'
RANKING_METRICS_DIR = PREDICTION_ROOT / 'ranking_metrics'
MODEL_DIR = PROJECT_ROOT / 'models' / 'lightgbm'

def parquet_files(path: Path) -> list[Path]:
    return sorted(path.rglob('*.parquet')) if path.exists() else []

def relation(path: Path) -> str:
    files = parquet_files(path)
    if not files:
        raise FileNotFoundError(f'No Parquet files found under {path}')
    quoted = ', '.join(
        "'" + file.as_posix().replace("'", "''") + "'" for file in files
    )
    return f'read_parquet([{quoted}], hive_partitioning=true)'

def inspect_dataset(path: Path) -> dict:
    files = parquet_files(path)
    return {
        'path': str(path),
        'exists': path.exists(),
        'parquet_files': len(files),
        'bytes': sum(file.stat().st_size for file in files),
        'rows': ds.dataset(str(path), format='parquet', partitioning='hive').count_rows()
        if files else 0,
    }

con = duckdb.connect()
con.execute("SET TimeZone='UTC'")
print('Project root:', PROJECT_ROOT)
print('Prediction root:', PREDICTION_ROOT)
print('Predictions available:', PREDICTIONS_DIR.exists())

## 2. File Manifest

A manifest confirms which prediction tables exist, how many Parquet files they contain, and their physical size.

In [ ]:
manifest = pd.DataFrame(
    [
        inspect_dataset(PREDICTIONS_DIR),
        inspect_dataset(TOP_PAGES_DIR),
        inspect_dataset(METRICS_DIR),
        inspect_dataset(RANKING_METRICS_DIR),
    ]
).assign(size_mb=lambda frame: frame['bytes'] / (1024 ** 2))
display(manifest)

## 3. Latest Run Metadata

`latest_run.json` records the origin hour, forecast hour, model manifest, exact feature order, quality-gate decision, and any metrics produced by the last scoring run.

In [ ]:
latest_run_path = PREDICTION_ROOT / 'latest_run.json'
if latest_run_path.exists():
    latest_run = json.loads(latest_run_path.read_text(encoding='utf-8'))
    display(pd.json_normalize(latest_run, sep='.'))
    print('Feature order:', latest_run.get('feature_columns'))
    print('Quality gate:', latest_run.get('quality_gate'))
else:
    print('latest_run.json is not available yet.')

## 4. Prediction Schema

Inspect the physical table contract before writing queries. The prediction table contains both the raw LightGBM output and the selected forecast after fallback logic.

In [ ]:
for name, path in {
    'predictions': PREDICTIONS_DIR,
    'research_top_pages': TOP_PAGES_DIR,
    'metrics': METRICS_DIR,
    'ranking_metrics': RANKING_METRICS_DIR,
}.items():
    if not parquet_files(path):
        print(f'{name}: no Parquet files')
        continue
    print(f'--- {name} ---')
    display(con.execute(f'DESCRIBE SELECT * FROM {relation(path)}').df())

## 5. Origin and Forecast Coverage

This shows which historical origin hours have been scored and which forecast hour each origin targets.

In [ ]:
prediction_relation = relation(PREDICTIONS_DIR)
coverage_sql = f'''
SELECT
    timestamp_hour AS origin_hour,
    forecast_hour,
    project, access_mode,
    count(*) AS rows,
    count(DISTINCT normalized_title) AS topics,
    sum(CASE WHEN actual_available THEN 1 ELSE 0 END) AS actual_rows,
    sum(CASE WHEN fallback_used THEN 1 ELSE 0 END) AS fallback_rows,
    sum(CASE WHEN forecast_method = 'unavailable' THEN 1 ELSE 0 END) AS unavailable_rows
FROM {prediction_relation}
GROUP BY 1, 2, 3, 4
ORDER BY 1
'''
display(con.execute(coverage_sql).df())

## 6. Sample Prediction Rows

Inspect the page-level forecast, current traffic, lag-1 baseline, predicted growth, and ranking fields.

In [ ]:
sample_sql = f'''
SELECT
    timestamp_hour, forecast_hour, source_project, project, access_mode, language, project_family,
    normalized_title, view_count, lag_1h_views,
    lightgbm_predicted_views, forecast_views, predicted_growth_rate,
    predicted_traffic_rank, predicted_growth_rank, forecast_method,
    fallback_used, fallback_reason, actual_available
FROM {prediction_relation}
ORDER BY timestamp_hour DESC, project, predicted_traffic_rank
LIMIT 30
'''
display(con.execute(sample_sql).df())

## 7. Missing and Invalid Prediction Checks

These checks identify null, NaN, negative, or non-finite model outputs and invalid rank values.

In [ ]:
quality_sql = f'''
SELECT
    count(*) AS rows,
    sum(CASE WHEN lightgbm_predicted_views IS NULL THEN 1 ELSE 0 END) AS null_model_predictions,
    sum(CASE WHEN lightgbm_predicted_views < 0 THEN 1 ELSE 0 END) AS negative_model_predictions,
    sum(CASE WHEN lightgbm_predicted_views IS NOT NULL AND NOT isfinite(lightgbm_predicted_views) THEN 1 ELSE 0 END) AS nonfinite_model_predictions,
    sum(CASE WHEN forecast_views IS NULL THEN 1 ELSE 0 END) AS null_selected_forecasts,
    sum(CASE WHEN forecast_views < 0 THEN 1 ELSE 0 END) AS negative_selected_forecasts,
    sum(CASE WHEN predicted_traffic_rank IS NULL OR predicted_traffic_rank <= 0 THEN 1 ELSE 0 END) AS invalid_traffic_ranks,
    sum(CASE WHEN predicted_growth_rank IS NULL OR predicted_growth_rank <= 0 THEN 1 ELSE 0 END) AS invalid_growth_ranks
FROM {prediction_relation}
'''
display(con.execute(quality_sql).df())

## 8. Fallback Coverage

`lag_1h_fallback` means the lag-1 forecast was selected. `unavailable` means both the model and lag-1 fallback were unavailable for that row.

In [ ]:
fallback_sql = f'''
SELECT
    timestamp_hour, project, access_mode,
    forecast_method,
    coalesce(fallback_reason, 'not_used') AS fallback_reason,
    count(*) AS rows,
    round(100.0 * count(*) / sum(count(*)) OVER (PARTITION BY timestamp_hour, project, access_mode), 2) AS pct_of_origin_project_access
FROM {prediction_relation}
GROUP BY 1, 2, 3, 4, 5
ORDER BY 1 DESC, 2, 3, 4, 5
'''
display(con.execute(fallback_sql).df())

In [ ]:
fallback_sample_sql = f'''
SELECT
    timestamp_hour, project, access_mode, normalized_title, view_count, lag_1h_views,
    lightgbm_predicted_views, forecast_views, forecast_method, fallback_reason
FROM {prediction_relation}
WHERE fallback_used OR forecast_method = 'unavailable'
ORDER BY timestamp_hour DESC, project, normalized_title
LIMIT 40
'''
display(con.execute(fallback_sample_sql).df())

## 9. Research Top Pages

The ranking table contains separate `predicted_traffic` and `predicted_growth` views.

In [ ]:
top_relation = relation(TOP_PAGES_DIR)
top_coverage_sql = f'''
SELECT forecast_hour, project, access_mode, ranking_type, count(*) AS rows,
       min(predicted_traffic_rank) AS min_traffic_rank,
       min(predicted_growth_rank) AS min_growth_rank
FROM {top_relation}
GROUP BY 1, 2, 3, 4
ORDER BY 1 DESC, 2, 3, 4
'''
display(con.execute(top_coverage_sql).df())

In [ ]:
traffic_top_sql = f'''
SELECT forecast_hour, project, access_mode, predicted_traffic_rank, normalized_title,
       view_count, forecast_views, predicted_growth_rate, forecast_method
FROM {top_relation}
WHERE ranking_type = 'predicted_traffic'
ORDER BY forecast_hour DESC, project, predicted_traffic_rank
LIMIT 60
'''
display(con.execute(traffic_top_sql).df())

In [ ]:
growth_top_sql = f'''
SELECT forecast_hour, project, access_mode, predicted_growth_rank, normalized_title,
       view_count, forecast_views, predicted_growth_rate, forecast_method
FROM {top_relation}
WHERE ranking_type = 'predicted_growth'
ORDER BY forecast_hour DESC, project, predicted_growth_rank
LIMIT 60
'''
display(con.execute(growth_top_sql).df())

## 10. Actual-vs-Prediction Metrics

Metrics are present only for origins whose next-hour target has been materialized. Re-running an origin replaces that origin's dynamic partitions rather than appending duplicate records.

In [ ]:
if parquet_files(METRICS_DIR):
    metrics_relation = relation(METRICS_DIR)
    metrics_sql = f'''
    SELECT evaluation_start_hour, evaluation_end_hour, forecast_method,
           evaluated_rows, mase_valid_rows, mase, nd, smape, msmape, scored_at_utc
    FROM {metrics_relation}
    ORDER BY scored_at_utc DESC, evaluation_start_hour DESC, forecast_method
    '''
    display(con.execute(metrics_sql).df())
else:
    print('No actual-vs-prediction metrics are available yet.')

if parquet_files(RANKING_METRICS_DIR):
    ranking_metrics_relation = relation(RANKING_METRICS_DIR)
    display(con.execute(f'''
        SELECT timestamp_hour, project, access_mode, k, forecast_coverage,
               ndcg_at_k, recall_at_k, top_k_overlap, spearman_rank_correlation
        FROM {ranking_metrics_relation}
        ORDER BY timestamp_hour DESC, project, access_mode, k
        LIMIT 100
    ''').df())
else:
    print('No ranking metrics are available yet.')

In [ ]:
actual_sample_sql = f'''
SELECT
    timestamp_hour, forecast_hour, project, access_mode, normalized_title,
    target_next_hour_views AS actual_views,
    lightgbm_predicted_views, lag_1h_views, forecast_views,
    abs(target_next_hour_views - forecast_views) AS selected_absolute_error,
    forecast_method, fallback_used
FROM {prediction_relation}
WHERE actual_available
ORDER BY selected_absolute_error DESC
LIMIT 40
'''
display(con.execute(actual_sample_sql).df())

## 11. Model Artifact Metadata

The saved metadata is the deployment contract for feature order, categories, target, model objective, and manifest identity.

In [ ]:
current_path = MODEL_DIR / 'current.json'
if current_path.exists():
    current = json.loads(current_path.read_text(encoding='utf-8'))
    version_dir = MODEL_DIR / current['path']
else:
    version_dir = MODEL_DIR
metadata_path = version_dir / 'metadata.json'
levels_path = version_dir / 'category_levels.json'
model_path = version_dir / 'model.txt'
print('Model file:', model_path, 'bytes:', model_path.stat().st_size if model_path.exists() else None)
if metadata_path.exists():
    model_metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    display(pd.json_normalize(model_metadata, sep='.'))
if levels_path.exists():
    category_levels = json.loads(levels_path.read_text(encoding='utf-8'))
    display(pd.DataFrame([
        {'column': column, 'levels': levels, 'level_count': len(levels)}
        for column, levels in category_levels.items()
    ]))

## 12. Research Checks

Use these checks before consuming the tables in a chart, API, or dashboard.

In [ ]:
checks = {
    'prediction_table_exists': PREDICTIONS_DIR.exists() and bool(parquet_files(PREDICTIONS_DIR)),
    'top_pages_table_exists': TOP_PAGES_DIR.exists() and bool(parquet_files(TOP_PAGES_DIR)),
    'model_metadata_exists': metadata_path.exists(),
    'category_levels_exists': levels_path.exists(),
    'model_file_exists': model_path.exists(),
    'all_selected_forecasts_nonnegative_or_null': con.execute(
        f"SELECT count(*) = 0 FROM {prediction_relation} WHERE forecast_views < 0"
    ).fetchone()[0] if PREDICTIONS_DIR.exists() and parquet_files(PREDICTIONS_DIR) else False,
}
display(pd.Series(checks, name='passed').to_frame())